In [1]:
%autosave 180
%load_ext autoreload
%autoreload 2

import matplotlib
#matplotlib.use('qtagg')
%matplotlib tk

#|
import os
import numpy as np
import time
import nidaqmx
from nidaqmx.constants import (AcquisitionType)  # https://nidaqmx-python.readthedocs.io/en/latest/constants.html
import numpy as np
import matplotlib.pyplot as plt
from nidaqmx.constants import TerminalConfiguration
import math

# 
from bmi import BMI, PlotROIs


Autosaving every 180 seconds


In [2]:
#####################################
#####################################
#####################################

#######################################
# FOR WINDOWS
# fname_root_path = r"D:\User Training"
# fname_fluorescence = r"D:\User Training\Readtest1\Image_001_001.raw"
# fname_freq =  r"F:\freq.npy"
# fname_rois = r"D:\User Training\rois.txt"

# FOR LINUX
fname_root_path = '/media/cat/4TB/donato/BSCOPE_tests/'
fname_fluorescence = os.path.join(fname_root_path, 
                                  'image_1000frames.raw')
fname_freq =  os.path.join(fname_root_path,
                           "freq.npy")
fname_rois = os.path.join(fname_root_path, 
                          "rois.txt")

# required for simulation mode
fname_ttl = os.path.join(fname_root_path,
                         "ttl_pulses.npy")


####################################################################### 			
################### DEFAULT PARAMTERS FOR BMI ######################### 			
####################################################################### 			
sampleRate_2P = 30
n_seconds_session = 10                          # number of seconds to run the BMI 
simulation_mode = True							# Run BMI in simulation mode (i.e. don't need Bscope input)

############################################################### 			
#################### INITIALIZE BMI ########################### 
############################################################### 	
bmi = BMI(simulation_mode,
		  fname_root_path,
		  fname_fluorescence,
		  fname_rois,
		  fname_freq,
          fname_ttl,
		  sampleRate_2P,
   		  n_seconds_session)

# for simulation mode we sometimes want to slow down the processing;
# ... not as necessary 
bmi.sleep_time_sec = 0.0001

# Flag to print out information from the proessing
bmi.verbose = False
bmi.verbose2 = False    # this displays the time it takes to copute ROI


 ROIS: ,  (10, 2)
   using square ROIs; TODO: use proper defined ROIs and cell masks ...
 shared memory rois traces:  (10, 300) psm_ca27edb5
 ttl counter initialized:  [0.] psm_015bd7e2


In [7]:
############################################################### 			
#################### INITIALIZE Plotting ###################### 
############################################################### 

# initialize plotting object
plotter = PlotROIs(bmi.shmem_rois_traces.name,
                   bmi.shmem_n_ttl.name,
                   bmi.rois_traces.shape)

# value to use for plotting; to separate different ROI traces
plotter.plot_y_scale = 1000


  Plotter loaded:  psm_ca27edb5  size:  (10, 300)
  loaded rois_traces:  (10, 300)
  Plotter loaded:  psm_015bd7e2
  loaded n_ttl:  [0.]


In [ ]:
############################################################### 			
#################### INITIALIZE TONE ########################## 
############################################################### 





In [ ]:

#
bmi.run_BMI()


In [ ]:
import sys
import time
from pyqtgraph.Qt import QtCore, QtGui
import numpy as np
import pyqtgraph as pg


class App(QtGui.QMainWindow):
    def __init__(self, parent=None):
        super(App, self).__init__(parent)

        #### Create Gui Elements ###########
        self.mainbox = QtGui.QWidget()
        self.setCentralWidget(self.mainbox)
        self.mainbox.setLayout(QtGui.QVBoxLayout())

        self.canvas = pg.GraphicsLayoutWidget()
        self.mainbox.layout().addWidget(self.canvas)

        self.label = QtGui.QLabel()
        self.mainbox.layout().addWidget(self.label)

        self.view = self.canvas.addViewBox()
        self.view.setAspectLocked(True)
        self.view.setRange(QtCore.QRectF(0,0, 100, 100))

        #  image plot
        self.img = pg.ImageItem(border='w')
        self.view.addItem(self.img)

        self.canvas.nextRow()
        #  line plot
        self.otherplot = self.canvas.addPlot()
        self.h2 = self.otherplot.plot(pen='y')


        #### Set Data  #####################

        self.x = np.linspace(0,50., num=100)
        self.X,self.Y = np.meshgrid(self.x,self.x)

        self.counter = 0
        self.fps = 0.
        self.lastupdate = time.time()

        #### Start  #####################
        self._update()

    def _update(self):

        self.data = np.sin(self.X/3.+self.counter/9.)*np.cos(self.Y/3.+self.counter/9.)
        self.ydata = np.sin(self.x/3.+ self.counter/9.)

        self.img.setImage(self.data)
        self.h2.setData(self.ydata)

        now = time.time()
        dt = (now-self.lastupdate)
        if dt <= 0:
            dt = 0.000000000001
        fps2 = 1.0 / dt
        self.lastupdate = now
        self.fps = self.fps * 0.9 + fps2 * 0.1
        tx = 'Mean Frame Rate:  {fps:.3f} FPS'.format(fps=self.fps )
        self.label.setText(tx)
        QtCore.QTimer.singleShot(1, self._update)
        self.counter += 1


if __name__ == '__main__':

    app = QtGui.QApplication(sys.argv)
    thisapp = App()
    thisapp.show()
    sys.exit(app.exec_())


In [ ]:
import sys
from PyQt5 import QtWidgets
import pyqtgraph as pg

from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.backends.backend_qt5agg import (NavigationToolbar2QT as NavigationToolbar)
from matplotlib.figure import Figure

import random
import numpy as np


class Window(QtWidgets.QDialog):
    def __init__(self, parent=None):
        super(Window, self).__init__(parent)

        # a figure instance to plot on
        self.fig = Figure()
        self.axes = self.fig.add_subplot(111)
        
        # this is the Canvas Widget that displays the `figure`
        # it takes the `figure` instance as a parameter to __init__
        #self.canvas = CustomFigureCanvas(parent=self)
        self.canvas = pg.GraphicsLayoutWidget()

        # this is the Navigation widget
        # it takes the Canvas widget and a parent
        #self.toolbar = NavigationToolbar(self.canvas, self)

        # Just some button connected to `plot` method
        #self.button = QtWidgets.QPushButton("Plot")
        #self.button.clicked.connect(self.plot)

        # set the layout
        layout = QtWidgets.QVBoxLayout()
        #layout.addWidget(self.toolbar)
        layout.addWidget(self.canvas)
        #layout.addWidget(self.button)
        self.setLayout(layout)

    def plot(self):

        self.xx = np.random.rand(50)
        self.yy = np.random.rand(50)
        self.axes.plot(self.xx, self.yy)

    def _update(self):
        for k in range(10):
            print ("self.y: ", self.yy)
            self.yy = self.yy+1
            self.axes.plot(self.xx, self.yy)
        
            self.fig.canvas.draw()
        
app = QtWidgets.QApplication(sys.argv)

main = Window()
main.show()
main.plot()
main._update()
sys.exit(app.exec_())

In [3]:
import sys 
from PyQt5.QtGui import *
from PyQt5.QtCore import *
import pyqtgraph as pg
import time
import numpy as np


class Screen(QMainWindow):
    def __init__(self):
        super(Screen, self).__init__()
        self.initUI()

    def initUI(self):
        self.x = np.array([1,2,3,4])
        self.y = np.array([1,4,9,16])
        self.plt = pg.PlotWidget()
        self.plot = self.plt.plot(self.x, self.y)

        #self.x = np.array([1,2,3,4])
        #self.y = np.array([1,4,9,16])
        #self.plt = pg.PlotWidget()
        #self.plt.plot(self.x, self.y)

        addBtn = QPushButton('Add Datapoint')
        addBtn.clicked.connect(self.addDataToPlot)
        addBtn.show()

        mainLayout = QVBoxLayout()
        mainLayout.addWidget(addBtn)
        mainLayout.addWidget(self.plt)

        self.mainFrame = QWidget()
        self.mainFrame.setLayout(mainLayout)
        self.setCentralWidget(self.mainFrame)

    def addDataToPlot(self):
        self.counter +=1 
        self.x = np.append(self.x, self.counter)
        self.y = np.append(self.y, self.counter)
        self.plot.setData(self.x, self.y)


app = QApplication(sys.argv)
window = Screen()
window.show()
#sys.exit(app.exec_())

#enter image description here

NameError: name 'QMainWindow' is not defined

In [18]:
# NOTES AND SCRATCH SPACE

In [13]:
data = np.load('/media/cat/4TB/donato/BSCOPE_tests/100k_frames_ttl_data.zip')
print (data['data'].shape)

ttl_pulses = data['data']
np.save('/media/cat/4TB/donato/BSCOPE_tests/ttl_pulses.npy', ttl_pulses)

#
plt.figure()
plt.plot(data['data'])
plt.show()

(3599999, 1)


In [23]:
fname_fluorescence = '/media/cat/4TB/donato/BSCOPE_tests/8105/20220309/Image_001_001.raw'
n_frames_to_be_acquired = 1000
data_Ca = np.memmap(fname_fluorescence, dtype='uint16', mode='r', 
									   shape=(n_frames_to_be_acquired,512,512))
print (data_Ca.shape)
    
data_Ca.tofile('/media/cat/4TB/donato/BSCOPE_tests/image_1000frames.raw')

(1000, 512, 512)
